# PRISM — End-to-End Turnkey Notebook

> One notebook, 5 main cells. Paste into Colab Pro+ (A100), connect Drive,
> run sequentially. Output: trained PRISM checkpoint + paper-ready figures.
>
> **Total runtime**: ~4 hours on A100 80GB (most spent in Step 2: teacher cache).
>
> **What this produces:**
> - `teacher_logits_webqsp.pt` — pre-computed Llama-3.1 teacher distributions
> - `prism_distill_best.pt` — Stage 1 LAS checkpoint
> - `prism_rl_improved_best.pt` — Stage 2 RL-tuned LAS checkpoint
> - `eval_metrics.json` — Hit@1, F1, ECE, Brier
> - `figure_abstention.pdf` — Risk-coverage curve (paper Figure)
> - `figure_reliability.pdf` — Calibration reliability diagram

---


## Step 0 — Setup (one-time)

Mount Drive, install deps, sync 6 PRISM files. Edit `PRISM_ROOT` if your folder is elsewhere.


In [ ]:
# ─── Mount Drive ───────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, sys, time, json
from pathlib import Path

# === EDIT THIS PATH IF NEEDED ===
DRIVE_BASE = '/content/drive/MyDrive/ThS/Research/colab'
PRISM_ROOT = f'{DRIVE_BASE}/graph_rag/prism'   # where the 6 prism .py files live
CACHE_DIR  = f'{DRIVE_BASE}/graph_rag/ckpt/prism'
OUT_DIR    = f'{DRIVE_BASE}/graph_rag/results/prism'
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

# Verify all 6 PRISM files exist
required = ['config.py', 'data.py', 'encoder.py', 'las.py',
            'controller.py', 'controller_batched.py',
            'distill.py', 'rl.py', 'metrics.py', 'evaluate.py',
            'teacher_llm.py', 'cache_teacher_logits.py',
            'improvements.py', '__init__.py']
missing = [f for f in required if not Path(f'{PRISM_ROOT}/{f}').exists()]
if missing:
    print('MISSING FILES — sync from local repo:')
    for f in missing: print(f'  - {f}')
    raise FileNotFoundError(f'{len(missing)} files missing in {PRISM_ROOT}')

# Add parent so `import prism` works
sys.path.insert(0, str(Path(PRISM_ROOT).parent))
print(f'\n✓ All {len(required)} PRISM files present')
print(f'  Drive base: {DRIVE_BASE}')
print(f'  PRISM root: {PRISM_ROOT}')


In [ ]:
# ─── Install deps (only if not already installed) ──────────────
import subprocess, importlib

def ensure(pkg, import_name=None):
    name = import_name or pkg
    try:
        importlib.import_module(name)
        print(f'  ✓ {name}')
    except ImportError:
        print(f'  installing {pkg}...')
        subprocess.run(['pip', 'install', '-q', pkg], check=True)
        importlib.import_module(name)

for p in [('torch', 'torch'),
          ('transformers', 'transformers'),
          ('accelerate', 'accelerate'),
          ('sentencepiece', 'sentencepiece'),
          ('datasets', 'datasets'),
          ('sentence-transformers', 'sentence_transformers'),
          ('tqdm', 'tqdm'),
          ('matplotlib', 'matplotlib')]:
    ensure(*p)

import torch
print(f'\n✓ torch={torch.__version__}  CUDA={torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU: {torch.cuda.get_device_name(0)}')

# Optional HF auth — needed only for gated Llama-3.1 (skip if using NousResearch mirror)
try:
    from google.colab import userdata
    if userdata.get('HF_TOKEN'):
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
        print('  ✓ HF_TOKEN set from Colab secrets')
except Exception:
    print('  (no HF_TOKEN in secrets — will use NousResearch fallback)')


## Step 1 — Configure PRISM

Set hyperparameters. **Don't change anything to reproduce paper numbers.**


In [ ]:
# ─── Import + configure ───────────────────────────────────────
import importlib
import prism
importlib.reload(prism)
from prism import CFG
from prism.config import Config

# Production settings (paper numbers)
CFG.seed = 42
CFG.device = 'cuda'
CFG.d_model = 256          # synthetic was 192; bump for real data
CFG.d_query = 384          # SBERT all-MiniLM-L6-v2 dim
CFG.n_heads = 4
CFG.n_layers = 4
CFG.max_paths = 50         # GNN-RAG returns up to this many
CFG.max_path_len = 3
CFG.n_candidates = 32
CFG.max_select_paths = 10
CFG.stopping_eps = 0.05
CFG.diversity_lambda = 0.3

# Stage 1 distillation
CFG.distill_epochs = 10
CFG.distill_batch_size = 8
CFG.distill_lr = 3e-4
CFG.distill_patience = 3

# Stage 2 RL
CFG.rl_steps = 500
CFG.rl_n_samples = 4
CFG.rl_lr = 5e-5
CFG.rl_calib_weight = 0.1
CFG.rl_w_coverage = 0.6
CFG.rl_w_efficiency = 0.2
CFG.rl_w_diversity = 0.2

# Output paths
CFG.out_dir = CACHE_DIR
TEACHER_CACHE = f'{CACHE_DIR}/teacher_logits_webqsp.pt'
DISTILL_CKPT  = f'{CACHE_DIR}/prism_distill_best.pt'
RL_CKPT       = f'{CACHE_DIR}/prism_rl_improved_best.pt'
METRICS_JSON  = f'{OUT_DIR}/eval_metrics.json'

print('Configured:')
for k in ['d_model', 'n_layers', 'max_select_paths', 'distill_epochs', 'rl_steps']:
    print(f'  {k} = {getattr(CFG, k)}')
print(f'\nTeacher cache → {TEACHER_CACHE}')
print(f'Distill ckpt → {DISTILL_CKPT}')
print(f'RL ckpt      → {RL_CKPT}')


## Step 2 — Pre-compute teacher logits (~60 min, ONE TIME)

Runs Llama-3.1-8B-Instruct on every (sample, subset) pair from WebQSP.
**Skip this step on re-runs if `teacher_logits_webqsp.pt` exists.**

### Why this is the most important step
Without real LLM teacher, PRISM's "distill from LLM" claim is not defensible.


In [ ]:
# ─── Build teacher cache ──────────────────────────────────────
from pathlib import Path

SKIP_TEACHER = Path(TEACHER_CACHE).exists() and Path(TEACHER_CACHE).stat().st_size > 1_000_000

if SKIP_TEACHER:
    print(f'✓ Existing teacher cache found at {TEACHER_CACHE}')
    print(f'  ({Path(TEACHER_CACHE).stat().st_size / 1e6:.1f} MB)')
    print(f'  Delete it manually if you want to rebuild.')
else:
    from prism.cache_teacher_logits import main_programmatic
    cache = main_programmatic(
        hf_name="rmanluo/RoG-webqsp",
        splits=("train",),
        out_path=TEACHER_CACHE,
        model_name="meta-llama/Llama-3.1-8B-Instruct",  # falls back to NousResearch
        subsets_per_sample=3,
        max_items=3000,                                  # cap for speed; set None for full
        cache_dir=f'{CACHE_DIR}/rog_cache',
        dtype='bf16',
    )
    print(f'\n✓ Cache built: {len(cache):,} entries at {TEACHER_CACHE}')

# Verify cache is loadable
import torch
loaded = torch.load(TEACHER_CACHE, map_location='cpu', weights_only=False)
sample_key = next(iter(loaded))
print(f'  Sample entry: key={sample_key}  shape={loaded[sample_key].shape}')
print(f'  Total entries: {len(loaded):,}')

# Free LLM memory before next step
import gc
gc.collect()
torch.cuda.empty_cache()


## Step 3 — Distill LAS from teacher (~30 min on A100)

Stage 1: Train LAS to mimic Llama-3.1's answer distribution.
**Loss should drop from ~1.5 to ~0.15 over 10 epochs.**


In [ ]:
# ─── Load data + LAS ──────────────────────────────────────────
from prism.data import RoGDataset
from prism.las import LAS
from prism.teacher_llm import CachedTeacherLLM

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Loading WebQSP train + validation...')
train_set = RoGDataset.from_hf(
    "rmanluo/RoG-webqsp", split="train",
    sbert_model="all-MiniLM-L6-v2",
    cache_dir=f'{CACHE_DIR}/rog_cache',
    cfg=CFG, max_items=3000,
)
eval_set = RoGDataset.from_hf(
    "rmanluo/RoG-webqsp", split="validation",
    sbert_model="all-MiniLM-L6-v2",
    cache_dir=f'{CACHE_DIR}/rog_cache',
    cfg=CFG, max_items=400,
)
CFG.n_relations = max(train_set.n_relations, eval_set.n_relations)
CFG.n_entities  = max(train_set.n_entities, eval_set.n_entities)
print(f'  train={len(train_set)}  eval={len(eval_set)}')
print(f'  n_relations={CFG.n_relations}  n_entities={CFG.n_entities}')

# Build LAS
las = LAS(CFG).to(device)
n_params = sum(p.numel() for p in las.parameters() if p.requires_grad)
print(f'\nLAS params: {n_params:,} = {n_params/1e6:.2f}M')

# Plug real teacher into distill module
import prism.distill as distill_mod
teacher = CachedTeacherLLM(cache_path=TEACHER_CACHE, strict=False, verbose=True)
distill_mod.TeacherLLM = lambda: teacher    # monkey-patch
print('\n✓ Real teacher plugged in')


In [ ]:
# ─── Run distillation ─────────────────────────────────────────
from prism.distill import train_distill

if Path(DISTILL_CKPT).exists():
    print(f'✓ Loading existing distill ckpt from {DISTILL_CKPT}')
    las.load_state_dict(torch.load(DISTILL_CKPT, map_location=device))
else:
    print('Starting Stage 1 distillation...')
    t0 = time.time()
    ckpt_path = train_distill(las, train_set, eval_set, cfg=CFG)
    dt = time.time() - t0
    print(f'\n✓ Distill complete in {dt/60:.1f} min')
    # Move to final location
    import shutil
    shutil.copy(ckpt_path, DISTILL_CKPT)
    print(f'  Saved → {DISTILL_CKPT}')

# Teacher cache stats
print(f'\nTeacher cache stats: {teacher.stats()}')


## Step 4 — RL fine-tune with improvements (~20 min on A100)

Stage 2: Apply REINFORCE SCST with calibration-shaped reward + entropy regularization.
**Reward should grow from baseline ~0.45 to ~0.55+ over 500 steps.**


In [ ]:
# ─── RL training with improvements ────────────────────────────
from prism.improvements import train_rl_improved

if Path(RL_CKPT).exists():
    print(f'✓ Loading existing RL ckpt from {RL_CKPT}')
    las.load_state_dict(torch.load(RL_CKPT, map_location=device))
else:
    print('Starting Stage 2 RL fine-tune...')
    t0 = time.time()
    ckpt_path = train_rl_improved(
        las, train_set, eval_set, cfg=CFG,
        reward_mode='calib_shaped',     # Improvement #1
        entropy_coef=0.02,               # Improvement #3
        use_batched=True,                # 100x speedup
        eval_every=100,
        log_every=25,
    )
    dt = time.time() - t0
    print(f'\n✓ RL complete in {dt/60:.1f} min')
    import shutil
    if Path(ckpt_path).exists():
        shutil.copy(ckpt_path, RL_CKPT)
        print(f'  Saved → {RL_CKPT}')


## Step 5 — Full evaluation + paper figures

Produces:
- `eval_metrics.json` — Hit@1, F1, ECE, Brier, mean #paths
- `figure_reliability.pdf` — reliability diagram
- `figure_abstention.pdf` — risk-coverage curve

**Use these directly in paper Section 5.**


In [ ]:
# ─── Run evaluation with batched controller ────────────────
from prism.controller_batched import deterministic_select
from prism.metrics import (hit_at_k, f1_score, expected_calibration_error,
                            brier_score, reliability_diagram)
from prism.improvements import (selective_abstention_eval, risk_coverage_curve,
                                 TemperatureScaler)
import numpy as np
from tqdm import tqdm

las.eval()
confidences, correctness, n_paths_list = [], [], []
f1s, k_chosen_list = [], []

print(f'Evaluating on {len(eval_set)} questions...')
t0 = time.time()
for i in tqdm(range(len(eval_set))):
    sample = eval_set[i]
    sample.query_emb = sample.query_emb.to(device)
    sample.paths = sample.paths.to(device)
    sample.path_to_candidate = sample.path_to_candidate.to(device)
    sample.candidate_gnn_scores = sample.candidate_gnn_scores.to(device)
    sample.candidate_ids = sample.candidate_ids.to(device)
    sample.gold_entities = sample.gold_entities.to(device)

    trace = deterministic_select(las, sample, CFG)
    hit = hit_at_k(trace.final_probs, sample.candidate_ids, sample.gold_entities, k=1)
    f1 = f1_score(trace.final_probs, sample.candidate_ids, sample.gold_entities, 0.05)

    confidences.append(float(trace.confidence.item()))
    correctness.append(int(hit))
    f1s.append(f1)
    n_paths_list.append(len(trace.selected_path_idx))
    k_chosen_list.append(trace.chosen_K)

dt = time.time() - t0
print(f'\n✓ Eval done in {dt:.1f}s  ({dt/len(eval_set)*1000:.1f} ms/q)')

# Core metrics
hit_at_1 = np.mean(correctness) * 100
f1_mean = np.mean(f1s) * 100
ece_raw = expected_calibration_error(confidences, correctness)
brier_raw = brier_score(confidences, correctness)
mean_paths = np.mean(n_paths_list)
mean_K = np.mean(k_chosen_list)

# Apply temperature scaling for calibration
print('\nFitting temperature scaling...')
ts = TemperatureScaler().to(device)
ts.fit(confidences, correctness, n_iter=200, lr=0.05)
calibrated_conf = [float(ts(torch.tensor([c], device=device)).item()) for c in confidences]
ece_calib = expected_calibration_error(calibrated_conf, correctness)
brier_calib = brier_score(calibrated_conf, correctness)
print(f'  Temperature T = {float(ts.T.item()):.3f}')
print(f'  ECE  raw={ece_raw:.4f}  calibrated={ece_calib:.4f}')
print(f'  Brier raw={brier_raw:.4f}  calibrated={brier_calib:.4f}')

# Selective abstention
abst = selective_abstention_eval(calibrated_conf, correctness,
                                  abstain_thresholds=(0.0, 0.3, 0.5, 0.7, 0.9))

# Print main results table
print('\n' + '='*60)
print('=== PRISM Main Results (WebQSP validation) ===')
print('='*60)
print(f'  Hit@1       : {hit_at_1:.2f}%')
print(f'  F1          : {f1_mean:.2f}%')
print(f'  Mean #paths : {mean_paths:.2f}')
print(f'  Mean K      : {mean_K:.2f}')
print(f'  ECE  (raw)  : {ece_raw:.4f}')
print(f'  ECE  (temp) : {ece_calib:.4f}')
print(f'  Brier (raw) : {brier_raw:.4f}')
print('\n=== Selective Abstention ===')
for r in abst:
    print(f'  thr={r["threshold"]:.1f}  coverage={r["coverage"]*100:5.1f}%  '
          f'hit={r["hit"]*100:5.2f}%')

# Save metrics
metrics = {
    'hit_at_1': hit_at_1, 'f1': f1_mean,
    'mean_paths': mean_paths, 'mean_K': mean_K,
    'ece_raw': ece_raw, 'ece_calibrated': ece_calib,
    'brier_raw': brier_raw, 'brier_calibrated': brier_calib,
    'temperature_T': float(ts.T.item()),
    'abstention': abst, 'n_eval': len(eval_set),
}
with open(METRICS_JSON, 'w') as f:
    json.dump(metrics, f, indent=2)
print(f'\n✓ Metrics saved → {METRICS_JSON}')


In [ ]:
# ─── Paper-ready figures ─────────────────────────────────────
import matplotlib
matplotlib.rcParams.update({'font.size': 12, 'figure.dpi': 150, 'font.family': 'serif'})
import matplotlib.pyplot as plt

# Figure 1: Reliability diagram (calibration)
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, (label, confs) in zip(axes, [('Raw confidence', confidences),
                                       ('Temperature-scaled', calibrated_conf)]):
    centers, accs, counts = reliability_diagram(confs, correctness, n_bins=10)
    valid = counts > 0
    width = 0.09
    ax.bar(centers[valid], accs[valid], width=width, alpha=0.6,
            color='#d62728', edgecolor='black', label='Empirical accuracy')
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect calibration')
    # Confidence line
    bin_conf = np.array([np.mean([c for c in confs if (centers[i] - 0.05) <= c < (centers[i] + 0.05)])
                          if counts[i] > 0 else np.nan for i in range(len(centers))])
    ax.scatter(centers[valid], bin_conf[valid], s=40, color='#1f77b4',
                marker='o', label='Mean confidence', zorder=5)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel('Predicted confidence'); ax.set_ylabel('Empirical accuracy')
    ece_show = expected_calibration_error(confs, correctness)
    ax.set_title(f'{label}\nECE = {ece_show:.4f}')
    ax.legend(loc='lower right', fontsize=10)
    ax.grid(alpha=0.3)
plt.suptitle(f'PRISM Calibration on WebQSP (n={len(eval_set)})', fontsize=13)
plt.tight_layout()
fig_path = f'{OUT_DIR}/figure_reliability.pdf'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.savefig(fig_path.replace('.pdf', '.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'✓ Saved {fig_path}')


# Figure 2: Risk-coverage curve (selective abstention)
fig, ax = plt.subplots(figsize=(7, 5))
rc = risk_coverage_curve(calibrated_conf, correctness, n_points=30)
xs = [r[0] for r in rc]
ys = [r[1] for r in rc]
ax.plot(xs, ys, 'o-', color='#d62728', linewidth=2, markersize=6,
         label='PRISM (calibrated)', zorder=4)

# Random baseline
xs_rand = np.linspace(0, 1, 50)
random_acc = np.mean(correctness)
ax.axhline(random_acc, color='gray', linestyle=':',
            label=f'No abstention (acc = {random_acc*100:.1f}%)')

# Mark threshold points
for thr in [0.3, 0.5, 0.7, 0.9]:
    kept = [i for i, c in enumerate(calibrated_conf) if c >= thr]
    if kept:
        cov = len(kept) / len(calibrated_conf)
        acc = np.mean([correctness[i] for i in kept])
        ax.scatter(cov, acc, s=200, edgecolor='black', facecolor='gold',
                   zorder=5, linewidth=1.5)
        ax.annotate(f'thr={thr}', (cov, acc),
                     xytext=(8, 8), textcoords='offset points', fontsize=10)

ax.set_xlabel('Coverage (fraction of questions answered)')
ax.set_ylabel('Hit@1 on answered subset')
ax.set_title(f'Selective Abstention — PRISM Risk-Coverage Curve\n'
              f'(WebQSP val, n={len(eval_set)}, post temp scaling)', fontsize=11)
ax.set_xlim(0, 1.02); ax.set_ylim(0.5, 1.02)
ax.grid(alpha=0.3)
ax.legend(loc='lower left', fontsize=10)
plt.tight_layout()
fig_path = f'{OUT_DIR}/figure_abstention.pdf'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.savefig(fig_path.replace('.pdf', '.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'✓ Saved {fig_path}')

print('\n' + '='*60)
print('=== ALL DONE ===')
print('='*60)
print(f'\nDeliverables:')
print(f'  metrics → {METRICS_JSON}')
print(f'  figure 1 → {OUT_DIR}/figure_reliability.pdf')
print(f'  figure 2 → {OUT_DIR}/figure_abstention.pdf')
print(f'\nUse these directly in paper Section 5.')
